In [1]:
from pathlib import Path
import random
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
from src.config import ANOMALY_SIGMA, DATA_PROCESSED, HEALTHY_RATIO, LSTM_BATCH, LSTM_EPOCHS, LSTM_HIDDEN, LSTM_LATENT, LSTM_LAYERS, LSTM_LR, LSTM_SEQ_LEN, MODELS_DIR, REPORTS_DIR, SEED, TRANSITION_RATIO
from src.models import compute_metrics
from src.models.lstm_autoencoder import LSTMAutoencoder, reconstruction_errors, train_autoencoder
from src.visualization.plots import plot_reconstruction_error

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
FIGURES_DIR = REPORTS_DIR / "figures"
CHECKPOINT_DIR = MODELS_DIR / "checkpoints"
METRICS_DIR = MODELS_DIR / "metrics"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)

In [4]:
femto_features = pd.read_parquet(DATA_PROCESSED / "femto_features_all_bearings.parquet")
bearing_features = femto_features[femto_features["bearing_id"] == "Bearing1_1"].copy()
bearing_features = bearing_features.sort_values(["file_idx", "window_idx"]).reset_index(drop=True)

In [5]:
file_count = int(bearing_features["file_idx"].max() + 1)
healthy_end = int(file_count * HEALTHY_RATIO)
anomaly_start = int(file_count * (HEALTHY_RATIO + TRANSITION_RATIO))
healthy_features = bearing_features[bearing_features["file_idx"] < healthy_end].copy()
anomaly_features = bearing_features[bearing_features["file_idx"] >= anomaly_start].copy()

In [6]:
feature_columns = bearing_features.select_dtypes(include=np.number).columns.drop(["file_idx", "window_idx"])
scaler = StandardScaler()
healthy_matrix = scaler.fit_transform(healthy_features[feature_columns].to_numpy())
anomaly_matrix = scaler.transform(anomaly_features[feature_columns].to_numpy())

In [7]:
healthy_sequences = np.stack([
    healthy_matrix[start:start + LSTM_SEQ_LEN]
    for start in range(len(healthy_matrix) - LSTM_SEQ_LEN + 1)
])
anomaly_sequences = np.stack([
    anomaly_matrix[start:start + LSTM_SEQ_LEN]
    for start in range(len(anomaly_matrix) - LSTM_SEQ_LEN + 1)
])

In [8]:
X_train_seq = healthy_sequences.astype(np.float32)
X_test_seq = np.concatenate([healthy_sequences, anomaly_sequences]).astype(np.float32)
y_test_seq = np.concatenate([
    np.zeros(len(healthy_sequences), dtype=int),
    np.ones(len(anomaly_sequences), dtype=int),
])

In [9]:
train_loader = DataLoader(
    TensorDataset(torch.from_numpy(X_train_seq)), batch_size=LSTM_BATCH, shuffle=True
)
test_loader = DataLoader(
    TensorDataset(torch.from_numpy(X_test_seq)), batch_size=LSTM_BATCH, shuffle=False
)

In [10]:
model = LSTMAutoencoder(
    input_dim=X_train_seq.shape[2],
    hidden_dim=LSTM_HIDDEN,
    latent_dim=LSTM_LATENT,
    num_layers=LSTM_LAYERS,
)
losses = train_autoencoder(model, train_loader, LSTM_EPOCHS, LSTM_LR, device)

Autoencoder epochs:   0%|          | 0/50 [00:00<?, ?it/s]

Autoencoder epochs:   2%|▏         | 1/50 [00:01<01:13,  1.50s/it]

Autoencoder epochs:   4%|▍         | 2/50 [00:02<00:59,  1.24s/it]

Autoencoder epochs:   6%|▌         | 3/50 [00:03<00:53,  1.14s/it]

Autoencoder epochs:   8%|▊         | 4/50 [00:04<00:50,  1.10s/it]

...

Autoencoder epochs:  94%|█████████▍| 47/50 [00:49<00:03,  1.04s/it]

Autoencoder epochs:  96%|█████████▌| 48/50 [00:50<00:02,  1.03s/it]

Autoencoder epochs:  98%|█████████▊| 49/50 [00:51<00:01,  1.05s/it]

Autoencoder epochs: 100%|██████████| 50/50 [00:52<00:00,  1.05s/it]

Autoencoder epochs: 100%|██████████| 50/50 [00:52<00:00,  1.05s/it]

In [11]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(np.arange(1, len(losses) + 1), losses, linewidth=1.4, label="Training loss")
ax.set_xlabel("Epoch")
ax.set_ylabel("MSE loss")
ax.set_title("LSTM Autoencoder Training Loss")
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES_DIR / "bearing1_1_lstm_ae_training_loss.png", dpi=150)
plt.close(fig)

In [12]:
train_errors = reconstruction_errors(model, train_loader, device)
test_errors = reconstruction_errors(model, test_loader, device)
threshold = float(np.mean(train_errors) + ANOMALY_SIGMA * np.std(train_errors))

In [13]:
plot_reconstruction_error(
    test_errors,
    FIGURES_DIR / "bearing1_1_lstm_ae_reconstruction_error.png",
    threshold=threshold,
    failure_idx=len(test_errors) - 1,
)

In [14]:
lstm_metrics = compute_metrics(y_test_seq, test_errors, threshold)
lstm_metrics_frame = pd.DataFrame([{"model": "LSTM Autoencoder", "threshold": threshold, **lstm_metrics}])
print(lstm_metrics_frame.to_string(index=False))

           model  threshold  roc_auc   pr_auc       f1  precision  recall  time_to_detection
LSTM Autoencoder   6.283337 0.999371 0.998085 0.970809   0.943275     1.0           0.731333


In [15]:
baseline_metrics = pd.read_csv(METRICS_DIR / "baseline_metrics.csv")
best_baseline = baseline_metrics.sort_values("f1", ascending=False).head(1)
comparison_frame = pd.concat([best_baseline, lstm_metrics_frame], ignore_index=True)
print(comparison_frame[["model", "roc_auc", "pr_auc", "f1", "precision", "recall"]].to_string(index=False))

           model  roc_auc   pr_auc       f1  precision   recall
     Rolling RMS 0.999998 0.999995 0.999554   1.000000 0.999109
LSTM Autoencoder 0.999371 0.998085 0.970809   0.943275 1.000000


In [16]:
torch.save(
    {
        "state_dict": model.state_dict(),
        "feature_columns": list(feature_columns),
        "scaler_mean": scaler.mean_,
        "scaler_scale": scaler.scale_,
        "threshold": threshold,
    },
    CHECKPOINT_DIR / "lstm_ae_bearing1_1.pt",
)

In [17]:
lstm_metrics_frame.to_csv(METRICS_DIR / "lstm_ae_metrics.csv", index=False)

The LSTM autoencoder should be judged against the frozen-threshold baselines, not against a supervised classifier. If it does not beat Rolling RMS on Bearing1_1, that is not a failure by itself: the baseline is exploiting a strong monotonic rise in vibration energy. The more important question is whether the reconstruction error remains useful when the same threshold and scaler are applied to other bearings and external datasets.